In [1]:
import pandas as pd
import numpy as np
from thefuzz import process, fuzz

In [2]:
df = pd.read_csv(r'E:\Data Science\FP using python\bestsellers_books.csv')
df_clean= df.copy()

In [3]:
df.describe()

,Name,Author,User Rating,Reviews,Price,Year,Genre
count,9475,9429,9470,9438,9470,9438,9414
unique,779,646,33,8219,100,22,18
top,Gone Girl,Roger Priddy,4.8,5069,12,2011,Non Fiction
freq,46,56,693,10,230,857,4563


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10500 entries, 0 to 10499
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Name         9475 non-null   object
 1   Author       9429 non-null   object
 2   User Rating  9470 non-null   object
 3   Reviews      9438 non-null   object
 4   Price        9470 non-null   object
 5   Year         9438 non-null   object
 6   Genre        9414 non-null   object
dtypes: object(7)
memory usage: 574.3+ KB


In [5]:
df.isnull().sum()

Name           1025
Author         1071
User Rating    1030
Reviews        1062
Price          1030
Year           1062
Genre          1086
dtype: int64

### Cleaning the 'Name' Column

In [6]:
df['Name'].nunique()

779

In [7]:
# Removes rows with missing 'Name' values
df_clean = df_clean.dropna(subset=['Name'])


In [8]:
df_clean['Name'] = df_clean['Name'].str.lower() # Converts 'Name' column to lowercase.
name_counts = df_clean['Name'].value_counts().reset_index() # Counts occurrences of each name.
name_counts.columns = ['Name', 'Count'] # Renames columns for clarity.
name_counts.to_csv('counts of books names.csv', index=False) # Saves counts to a CSV file.

In [9]:
ref_df= pd.read_csv(r'E:\Data Science\FP using python\reference_list.csv') # Loads book names reference list 

In [10]:
#  Convert the 'Name' column in reference  dataframe to lowercase for consistent comparison
ref_names = ref_df['Name'].str.lower().unique()


# Function to correct the name based on the best match from the reference list
def correct_name(name, choices, threshold=80):
    match, score = process.extractOne(name, choices)
    if score >= threshold:
        return match
    else:
        return name

# Apply the correction to the 'Name' column in the book data
df_clean['Name_corrected'] = df_clean['Name'].apply(lambda x: correct_name(x, ref_names))


In [11]:
name_counts = df_clean['Name_corrected'].value_counts().reset_index() # Counts occurrences of corrected names.
name_counts.columns = ['Name', 'Count'] # Renames columns for clarity.
name_counts.to_csv('cleaned books names.csv', index=False) # Saves the counts of cleaned book names to a CSV.

In [12]:
manual_corrections = {
    'lief': 'life',  # Defines manual corrections for common misspellings.
    'gtus': 'guts',
    'ugts': 'guts',
    'gust': 'guts'
}

# Apply the correction
df_clean['Name_corrected'] = df_clean['Name_corrected'].replace(manual_corrections) # Applies the defined manual corrections to the 'Name_corrected' column.

In [13]:
name_counts = df_clean['Name_corrected'].value_counts().reset_index() # Counts occurrences of corrected names.
name_counts.columns = ['Name', 'Count'] # Renames columns for clarity.
name_counts.to_csv('cleaned books names.csv', index=False) # Saves the counts of cleaned book names to a CSV.

In [14]:
df_clean['Name_corrected'] = df_clean['Name_corrected'].replace({
    'eclipse (twilight)': 'eclipse (twilight sagas)', # Corrects specific book titles for consistency.
    'the girl who played with fire (millennium)': 'the girl who played with fire (millennium series)'
})

In [15]:
name_counts = df_clean['Name_corrected'].value_counts().reset_index() # Counts occurrences of corrected names.
name_counts.columns = ['Name', 'Count'] # Renames columns for clarity.
name_counts.to_csv('cleaned books names.csv', index=False) # Saves the counts of cleaned book names to a CSV.

In [16]:
df_clean.drop(columns=['Name'], inplace=True) # Removes the original 'Name' column.
df_clean = df_clean[['Name_corrected'] + [col for col in df_clean.columns if col != 'Name_corrected']] # Reorders columns, placing 'Name_corrected' first.
df_clean['Name_corrected'] = df_clean['Name_corrected'].str.title() # Converts 'Name_corrected' to title case.

### Cleaning the 'Author' Column

In [17]:
df_clean['Author'] = df_clean['Author'].str.lower() # Converts 'Author' column to lowercase.
name_counts = df_clean['Author'].value_counts().reset_index() # Counts occurrences of each author.
name_counts.columns = ['Author', 'Count'] # Renames columns for clarity.
name_counts.to_csv('counts author.csv', index=False) # Saves author counts to a CSV file.

In [18]:
ref_df= pd.read_csv(r'Authors_list.csv') # Loads a list of authors from a CSV file.

In [19]:
#  Convert the 'Author' column in reference  dataframe to lowercase for consistent comparison
ref_names = ref_df['Author'].str.lower().unique()

# Function to correct the name based on the best match from the reference list
def correct_name(name, choices, threshold=85):
    if not isinstance(name, str) or pd.isnull(name):
        return name
    result = process.extractOne(name, choices)
    if result:
        match, score = result[:2] 
        return match if score >= threshold else name
    return name


# Apply the correction to the 'Author' column in the book data
df_clean['Author_corrected'] = df_clean['Author'].apply(lambda x: correct_name(x, ref_names))


In [20]:
name_counts = df_clean['Author_corrected'].value_counts().reset_index() # Counts occurrences of corrected author names.
name_counts.columns = ['Name', 'Count'] # Renames columns for clarity.
name_counts.to_csv('cleaned authors names.csv', index=False) # Saves the counts of cleaned author names to a CSV.

In [21]:
manual_corrections = {
    'gallpu': 'gallup',
    'george r. r. martin': 'george r.r. martin',
    'j. k. rowling': 'j.k. rowling',
    'kd': 'dk',
    'jj smith': 'j.j. smith'
}

# Apply the correction
df_clean['Author_corrected'] = df_clean['Author_corrected'].replace(manual_corrections)

In [22]:
name_counts = df_clean['Author_corrected'].value_counts().reset_index()
name_counts.columns = ['Name', 'Count']
name_counts.to_csv('cleaned authors names.csv', index=False)

In [23]:
 # Removes the original 'Author' column.
df_clean.drop(columns=['Author'], inplace=True)
# Reorders columns, placing 'Name_corrected' and 'Author_corrected' first.
df_clean = df_clean[['Name_corrected', 'Author_corrected'] + [col for col in df_clean.columns if col not in ['Name_corrected', 'Author_corrected']]] 
# Converts 'Author_corrected' to title case.
df_clean['Author_corrected'] = df_clean['Author_corrected'].str.title() 

### Remove non-numeric symbols from columns and convert types

In [24]:
df['User Rating'].unique()

array(['4.7', '4.6', '4.8', '4.4', '4.5', nan, '3.9', '4.3', '4.2', '4.9',
       '$4.50', '$4.70', '$4.40', '3.8', '$4.60', '4', '4.1', '$4.90',
       '$4.10', '$4.80', '3.3', '$4.20', '3.5', '3.7', '$3.60', '3.6',
       '5', '$4.00', '$3.70', '$3.80', '$4.30', '$3.90', '$5.00', '$3.50'],
      dtype=object)

In [25]:
df_clean['User Rating'] = df_clean['User Rating'].replace(r'\$', '', regex=True).astype(float) # Removes '$' from 'User Rating' and converts to float.
df_clean['User Rating'].unique() # Displays unique values in 'User Rating' column.

array([4.7, 4.4, 4.6, 4.5, nan, 3.9, 4.3, 4.2, 4.8, 4.9, 3.8, 4. , 4.1,
       3.3, 3.5, 3.7, 3.6, 5. ])

In [26]:
df['Reviews'].unique()

array(['17350', '2052', '18979', ..., '$1,651.00', '35437', '$7,855.00'],
      shape=(8220,), dtype=object)

In [27]:
df_clean['Reviews'] = df_clean['Reviews'].replace(r'[\$,]', '', regex=True).astype(float) # Removes '$' and ',' from 'Reviews' and converts to float.
df_clean['Reviews'].unique() # Displays unique values in 'Reviews' column.

array([17350., 18979., 21424., ..., 24671., 29591., 46700.], shape=(7391,))

In [28]:
df['Price'].unique()

array(['8', '$22.00', '15', '6', '12', '11', '30', '3', '2', nan, '5',
       '17', '4', '$8.00', '13', '14', '$14.00', '9', '$11.00', '24',
       '21', '0', '18', '28', '16', '10', '105', '22', '20', '1', '7',
       '$7.00', '32', '$5.00', '19', '54', '52', '25', '$6.00', '27',
       '$9.00', '46', '$46.00', '$17.00', '39', '53', '$15.00', '40',
       '36', '$82.00', '23', '$16.00', '42', '43', '38', '49', '34', '47',
       '37', '35', '26', '31', '33', '$21.00', '41', '29', '45', '44',
       '48', '$41.00', '$26.00', '$49.00', '$47.00', '$25.00', '$31.00',
       '$20.00', '$44.00', '$35.00', '$28.00', '$39.00', '$24.00',
       '$37.00', '$38.00', '$30.00', '$10.00', '$29.00', '$27.00',
       '$40.00', '$19.00', '$32.00', '$12.00', '$13.00', '$48.00',
       '$45.00', '$33.00', '$36.00', '$42.00', '$23.00', '$34.00',
       '$43.00', '$18.00'], dtype=object)

In [29]:
df_clean['Price (USD)'] = df_clean['Price'].replace(r'[\$,]', '', regex=True).astype(float) # Cleans 'Price' column and converts to float (USD).
df_clean.drop(columns=['Price'], inplace=True) # Removes the original 'Price' column.
df_clean['Price (USD)'].unique() # Displays unique values in 'Price (USD)' column.

array([  8.,  15.,   6.,  11.,  30.,   3.,   2.,  nan,   5.,  17.,  13.,
        14.,   9.,  24.,  21.,   4.,  18.,  28.,  16.,  10., 105.,   0.,
        12.,  20.,   1.,   7.,  32.,  19.,  54.,  22.,  52.,  25.,  27.,
        46.,  39.,  53.,  40.,  36.,  82.,  23.,  42.,  43.,  38.,  49.,
        34.,  47.,  37.,  35.,  26.,  31.,  33.,  41.,  29.,  45.,  44.,
        48.])

In [30]:
df['Year'].unique()

array(['2016', '2011', '2018', '2017', '2019', '2014', nan, '2010',
       '2009', '2015', '2013', '$2,015.00', '2012', '$2,019.00',
       '$2,018.00', '$2,014.00', '$2,017.00', '$2,012.00', '$2,011.00',
       '$2,010.00', '$2,016.00', '$2,013.00', '$2,009.00'], dtype=object)

In [31]:
df_clean['Year'] = df_clean['Year'].replace(r'[\$,]', '', regex=True).astype(float).astype('Int64') # Cleans 'Year' column and converts to integer.
df_clean['Year'].unique()# Displays unique values in 'Year' column.

<IntegerArray>
[2016, 2018, 2017, 2011, 2014, <NA>, 2010, 2009, 2015, 2013, 2012, 2019]
Length: 12, dtype: Int64

In [32]:
df['Genre'].unique()

array(['Non Fiction', 'Fiction', 'Fitcion', 'Ficiton', nan, 'Non Fcition',
       'Fictoin', 'oNn Fiction', 'No nFiction', 'Non Fitcion',
       'Non iFction', 'Fcition', 'Nno Fiction', 'iFction', 'Fictino',
       'Non Ficiton', 'Non Fictoin', 'NonF iction', 'Non Fictino'],
      dtype=object)

In [33]:
valid_genres = ['Fiction', 'Non Fiction'] # Defines a list of valid genres.
df_clean['Genre_clean'] = df['Genre'].apply(
    lambda x: process.extractOne(x, valid_genres)[0] if x and pd.notnull(x) and str(x).strip() != '' and process.extractOne(x, valid_genres)[1] >= 85 else x
) # Cleans 'Genre' column using fuzzy matching to valid genres.
df_clean.drop(columns=['Genre'], inplace=True) # Removes the original 'Genre' column.
df_clean['Genre_clean'].unique() # Displays unique values in 'Genre_clean' column.

array(['Non Fiction', 'Fiction', nan], dtype=object)

In [34]:
counts = df_clean['Genre_clean'].value_counts()[['Fiction', 'Non Fiction']]
print(counts)


Genre_clean
Fiction        4168
Non Fiction    4318
Name: count, dtype: int64


In [35]:
df_clean.to_csv('output_of_cleaned_books_dataset.csv', index=False, encoding='utf-8') # Saves the dataset after the initial cleaning phase .

In [36]:
df_clean.describe(include='all')


,Name_corrected,Author_corrected,User Rating,Reviews,Year,Price (USD),Genre_clean
count,9475,8506,8546.000000,8528.000000,8527.0,8531.000000,8486
unique,348,246,NaN,NaN,<NA>,NaN,2
top,The 5 Love Languages: The Secret To Love That ...,George R.R. Martin,NaN,NaN,<NA>,NaN,Non Fiction
freq,70,61,NaN,NaN,<NA>,NaN,4318
mean,NaN,NaN,4.273017,24085.337828,2013.977249,26.433478,NaN
std,NaN,NaN,0.435644,14613.635735,3.150714,13.292123,NaN
min,NaN,NaN,3.300000,58.000000,2009.0,0.000000,NaN
25%,NaN,NaN,3.900000,11231.750000,2011.0,15.000000,NaN
50%,NaN,NaN,4.300000,23644.500000,2014.0,26.000000,NaN
75%,NaN,NaN,4.700000,36430.000000,2017.0,38.000000,NaN


### Handling Duplicates

In [37]:
df_clean.duplicated().sum()

np.int64(447)

In [38]:
duplicate_all_subset = df_clean[df_clean.duplicated(keep=False)] # Identifies all duplicate rows across all columns.
duplicate_all_subset.sort_values(list(df_clean.columns)).head(10) # Sorts and displays the first 10 duplicate rows for inspection.

,Name_corrected,Author_corrected,User Rating,Reviews,Year,Price (USD),Genre_clean
7423,10-Day Green Smoothie Cleanse,Steve Harvey,5.0,2073.0,<NA>,40.0,Non Fiction
10056,10-Day Green Smoothie Cleanse,Steve Harvey,5.0,2073.0,<NA>,40.0,Non Fiction
7601,11/22/63: A Novel,Lysa Terkeurst,3.8,10170.0,2014,7.0,Non Fiction
10383,11/22/63: A Novel,Lysa Terkeurst,3.8,10170.0,2014,7.0,Non Fiction
3794,1984 (Signet Classics),Drew Daywalt,4.6,12460.0,<NA>,21.0,Non Fiction
10139,1984 (Signet Classics),Drew Daywalt,4.6,12460.0,<NA>,21.0,Non Fiction
7873,1984 (Signet Classics),Hillary Rodham Clinton,4.7,NaN,2014,28.0,Non Fiction
10353,1984 (Signet Classics),Hillary Rodham Clinton,4.7,NaN,2014,28.0,Non Fiction
1946,A Dance With Dragons (A Song Of Ice And Fire),Charles Krauthammer,4.5,40452.0,2019,10.0,NaN
10003,A Dance With Dragons (A Song Of Ice And Fire),Charles Krauthammer,4.5,40452.0,2019,10.0,NaN


In [39]:
duplicate_subset = df_clean[df_clean.duplicated(subset=['Name_corrected', 'Author_corrected', 'Year'], keep=False)] # Finds duplicates based on name, author, and year.
duplicate_subset.sort_values(['Name_corrected', 'Author_corrected', 'Year']).head(20) # Sorts and shows the first 20 duplicates for these columns.

,Name_corrected,Author_corrected,User Rating,Reviews,Year,Price (USD),Genre_clean
7423,10-Day Green Smoothie Cleanse,Steve Harvey,5.0,2073.0,<NA>,40.0,Non Fiction
10056,10-Day Green Smoothie Cleanse,Steve Harvey,5.0,2073.0,<NA>,40.0,Non Fiction
6014,10-Day Green Smoothie Cleanse,NaN,4.8,NaN,2017,40.0,Non Fiction
9241,10-Day Green Smoothie Cleanse,NaN,4.1,8203.0,2017,25.0,Non Fiction
9335,10-Day Green Smoothie Cleanse,NaN,4.8,47343.0,2017,10.0,Fiction
7601,11/22/63: A Novel,Lysa Terkeurst,3.8,10170.0,2014,7.0,Non Fiction
10383,11/22/63: A Novel,Lysa Terkeurst,3.8,10170.0,2014,7.0,Non Fiction
4273,11/22/63: A Novel,NaN,4.6,33256.0,2015,36.0,Fiction
6942,11/22/63: A Novel,NaN,3.8,23326.0,2015,12.0,Fiction
3794,1984 (Signet Classics),Drew Daywalt,4.6,12460.0,<NA>,21.0,Non Fiction


In [40]:
duplicate_subset.duplicated().sum()

np.int64(447)

In [41]:
df_copy = pd.read_csv(r'E:\Data Science\FP using python\cleaned dataset v.1.csv') # Loads the dataset after manual author column adjustments.

In [42]:
def get_mode_or_nan(series): # Defines a helper function to get the mode or NaN if empty.
    mode = series.mode()     # Calculates the mode of a series.
    return mode[0] if not mode.empty else np.nan # Returns the mode or NaN.

df_copy['User Rating'] = df_copy['User Rating'].fillna(df_copy['User Rating'].median()).round(2) # Fills missing 'User Rating' with the median and rounds.

df_copy['Reviews'] = df_copy['Reviews'].fillna(0) # Fills missing 'Reviews' with 0.
df_copy['Price (USD)'] = df_copy['Price (USD)'].fillna(df_clean['Price (USD)'].median()).round(2) # Fills missing 'Price (USD)' with the median and rounds.

df_copy['Genre_clean'] = df_copy['Genre_clean'].fillna(get_mode_or_nan(df_copy['Genre_clean'])) # Fills missing 'Genre_clean' with its mode.
df_copy['Year'] = df_copy['Year'].fillna(get_mode_or_nan(df_copy['Year'])) # Fills missing 'Year' with its mode.

print(df_copy.isnull().sum()) # Prints the count of missing values per column.

Name_corrected      0
Author_corrected    0
User Rating         0
Reviews             0
Year                0
Price (USD)         0
Genre_clean         0
dtype: int64


In [43]:
duplicate_subset = df_copy[df_copy.duplicated(subset=['Name_corrected', 'Year'], keep=False)] # Finds duplicates based on corrected name and year.
duplicate_subset.sort_values(['Name_corrected', 'Year']).head(50) # Sorts and displays the first 50 duplicates for these columns.

,Name_corrected,Author_corrected,User Rating,Reviews,Year,Price (USD),Genre_clean
7,10-Day Green Smoothie Cleanse,J.J. Smith,3.8,0.0,2010.0,38.0,Fiction
17,10-Day Green Smoothie Cleanse,J.J. Smith,4.0,26952.0,2010.0,34.0,Fiction
6,10-Day Green Smoothie Cleanse,J.J. Smith,3.7,0.0,2011.0,26.0,Fiction
9,10-Day Green Smoothie Cleanse,J.J. Smith,4.2,14591.0,2011.0,25.0,Non Fiction
11,10-Day Green Smoothie Cleanse,J.J. Smith,4.9,35744.0,2011.0,26.0,Non Fiction
13,10-Day Green Smoothie Cleanse,J.J. Smith,4.1,0.0,2011.0,34.0,Non Fiction
18,10-Day Green Smoothie Cleanse,J.J. Smith,5.0,2073.0,2011.0,40.0,Non Fiction
26,10-Day Green Smoothie Cleanse,J.J. Smith,5.0,2073.0,2011.0,40.0,Non Fiction
1,10-Day Green Smoothie Cleanse,J.J. Smith,4.9,37106.0,2014.0,43.0,Fiction
5,10-Day Green Smoothie Cleanse,J.J. Smith,4.3,8070.0,2014.0,45.0,Non Fiction


In [44]:
df_deduplicated = df_copy.groupby(['Name_corrected', 'Year'], as_index=False).agg({
    'Author_corrected': lambda x: get_mode_or_nan(x), # Groups by name and year, aggregates author by mode.
    'User Rating': 'median', # Aggregates 'User Rating' by median.
    'Reviews': 'sum', # Aggregates 'Reviews' by sum.
    'Price (USD)': 'median', # Aggregates 'Price (USD)' by median.
    'Genre_clean': lambda x: get_mode_or_nan(x) # Aggregates 'Genre_clean' by mode.
})
df_deduplicated['User Rating'] = df_deduplicated['User Rating'].round(2) # Rounds 'User Rating' to two decimal places.
df_deduplicated['Price (USD)'] = df_deduplicated['Price (USD)'].round(2) # Rounds 'Price (USD)' to two decimal places.

df_deduplicated.head(20) # Displays the first 20 rows of the deduplicated DataFrame.

,Name_corrected,Year,Author_corrected,User Rating,Reviews,Price (USD),Genre_clean
0,10-Day Green Smoothie Cleanse,2009.0,J.J. Smith,4.10,39150.0,41.0,Fiction
1,10-Day Green Smoothie Cleanse,2010.0,J.J. Smith,3.90,26952.0,36.0,Fiction
2,10-Day Green Smoothie Cleanse,2011.0,J.J. Smith,4.55,54481.0,30.0,Non Fiction
3,10-Day Green Smoothie Cleanse,2014.0,J.J. Smith,4.25,88283.0,40.5,Fiction
4,10-Day Green Smoothie Cleanse,2015.0,J.J. Smith,3.70,48242.0,24.0,Fiction
5,10-Day Green Smoothie Cleanse,2016.0,J.J. Smith,4.35,57700.0,20.0,Fiction
6,10-Day Green Smoothie Cleanse,2017.0,J.J. Smith,4.70,86481.0,26.0,Non Fiction
7,10-Day Green Smoothie Cleanse,2018.0,J.J. Smith,4.30,27047.0,49.0,Fiction
8,10-Day Green Smoothie Cleanse,2019.0,J.J. Smith,3.70,22505.0,13.0,Fiction
9,11/22/63: A Novel,2010.0,Stephen King,4.40,91705.0,17.5,Fiction


In [45]:
df_deduplicated.duplicated().sum()

np.int64(0)

In [46]:
df_deduplicated.duplicated(subset=['Name_corrected', 'Year']).sum()


np.int64(0)

In [ ]:
book_genre_map = df_deduplicated.groupby('Name_corrected')['Genre_clean'].agg(
    lambda x: x.mode()[0] if len(x.mode()) == 1 else x.iloc[0] 
)

df_deduplicated['Genre_corrected'] = df_deduplicated['Name_corrected'].map(book_genre_map)


In [48]:

df_deduplicated.iloc[105:117] # Displays rows 50 to 79 of the DataFrame with corrected genres.

,Name_corrected,Year,Author_corrected,User Rating,Reviews,Price (USD),Genre_clean,Genre_corrected
105,A Stolen Life: A Memoir,2009.0,Jaycee Dugard,4.35,56224.0,28.0,Fiction,Fiction
106,A Stolen Life: A Memoir,2010.0,Jaycee Dugard,3.60,0.0,47.0,Non Fiction,Fiction
107,A Stolen Life: A Memoir,2011.0,Jaycee Dugard,4.50,167878.0,29.0,Non Fiction,Fiction
108,A Stolen Life: A Memoir,2012.0,Jaycee Dugard,4.30,0.0,34.0,Non Fiction,Fiction
109,A Stolen Life: A Memoir,2013.0,Jaycee Dugard,4.50,77372.0,33.0,Non Fiction,Fiction
110,A Stolen Life: A Memoir,2014.0,Jaycee Dugard,4.30,94403.0,20.5,Fiction,Fiction
111,A Stolen Life: A Memoir,2015.0,Jaycee Dugard,4.30,76730.0,37.0,Fiction,Fiction
112,A Stolen Life: A Memoir,2016.0,Jaycee Dugard,4.50,57477.0,42.0,Fiction,Fiction
113,A Stolen Life: A Memoir,2017.0,Jaycee Dugard,4.00,48739.0,29.0,Non Fiction,Fiction
114,A Stolen Life: A Memoir,2019.0,Jaycee Dugard,4.65,45918.0,30.0,Fiction,Fiction


In [49]:
df_deduplicated.drop(columns=['Genre_clean'], inplace=True)


In [50]:
df_deduplicated.rename(columns={
    'Name_corrected': 'Name',
    'Author_corrected': 'Author',
    'Genre_corrected': 'Genre',
    'Price (USD)': 'Price ($)'
}, inplace=True)


In [51]:
df_deduplicated.head(10)

,Name,Year,Author,User Rating,Reviews,Price ($),Genre
0,10-Day Green Smoothie Cleanse,2009.0,J.J. Smith,4.10,39150.0,41.0,Fiction
1,10-Day Green Smoothie Cleanse,2010.0,J.J. Smith,3.90,26952.0,36.0,Fiction
2,10-Day Green Smoothie Cleanse,2011.0,J.J. Smith,4.55,54481.0,30.0,Fiction
3,10-Day Green Smoothie Cleanse,2014.0,J.J. Smith,4.25,88283.0,40.5,Fiction
4,10-Day Green Smoothie Cleanse,2015.0,J.J. Smith,3.70,48242.0,24.0,Fiction
5,10-Day Green Smoothie Cleanse,2016.0,J.J. Smith,4.35,57700.0,20.0,Fiction
6,10-Day Green Smoothie Cleanse,2017.0,J.J. Smith,4.70,86481.0,26.0,Fiction
7,10-Day Green Smoothie Cleanse,2018.0,J.J. Smith,4.30,27047.0,49.0,Fiction
8,10-Day Green Smoothie Cleanse,2019.0,J.J. Smith,3.70,22505.0,13.0,Fiction
9,11/22/63: A Novel,2010.0,Stephen King,4.40,91705.0,17.5,Fiction


In [ ]:
df_deduplicated.to_csv("Bestseller  Books.csv", index=False, encoding='utf-8') # Saves the final cleaned and deduplicated dataset to a CSV file.